In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../..')

In [ ]:
from Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Pro.DataPrep import Prep_Map, Output_Map
from College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map

data_prep = Combined_Data_Prep(
    Prep_Map.base_prep_map, 
    Output_Map.base_output_map,
    C_Prep_Map.college_base_prep_map,
    C_Output_Map.college_output_map)

hitter_io_list = data_prep.Generate_IO_Hitters(is_training=True)

In [ ]:
from Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets

train_dataset, test_dataset = Create_Test_Train_Datasets(
    player_list=hitter_io_list, 
    is_hitter=True)

In [ ]:
from Combined.Tuning.ProWar import objective
from functools import partial

objective_func = partial(
    objective,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    data_prep=data_prep,
    is_hitter=True,
    max_repeats=2,
)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="minimize",
    load_if_exists=True,
    study_name="Pro_WAR_Tuning",
    storage="sqlite:///Pro_Tune.db"
)

In [ ]:
study.optimize(objective_func, n_trials=300, show_progress_bar=True)

In [ ]:
print(f"Final best value after refinement: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
import optuna.visualization as vis
vis.plot_param_importances(study).show()

In [ ]:
vis.plot_optimization_history(study).show()

In [ ]:
vis.plot_slice(study).show()

In [ ]:
vis.plot_contour(study, params=["hidden_size", "dropout"]).show()